# CUDA attention kernels: build, test, benchmark

Runtime > Change runtime type > GPU. A T4 runs v1 to v3 and the tests; the FlashAttention-2 baseline needs an Ampere or newer GPU (A100, L4).

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

## 1. Get the code
Upload the repo zip to the Colab file pane, or clone it from GitHub.

In [ ]:
# !git clone https://github.com/b4batunde/cuda-attention.git
!unzip -q -o cuda-attention.zip 2>/dev/null; %cd cuda-attention

## 2. Build the extension (2 to 4 minutes)

In [ ]:
!pip install -q ninja pytest matplotlib
!MAX_JOBS=2 pip install -q -e . --no-build-isolation

## 3. Baseline: FlashAttention-2 (Ampere+ only)
Use a prebuilt wheel that matches your torch and CUDA versions if the source build is slow.

In [ ]:
import torch
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -q flash-attn --no-build-isolation
else:
    print('Turing GPU: skipping flash-attn; benchmark will mark flash2 as unavailable')

## 4. Correctness first

In [ ]:
!python -m pytest -q tests

## 5. Benchmark sweep and plots

In [ ]:
!python bench/benchmark.py
import glob
csv_path = sorted(glob.glob('results/measured/bench_*.csv'))[-1]
!python bench/plot.py {csv_path}

In [ ]:
from IPython.display import Image, Markdown, display
stem = csv_path[:-4]
display(Image(stem + '_throughput.png'))
display(Image(stem + '_memory.png'))
display(Markdown(open(stem + '_summary.md').read()))

## 6. Profile (needs Nsight Compute; preinstalled on most cloud GPU images, not on Colab)

In [ ]:
# !bash bench/profile.sh

## 7. Save results
Download `results/` before the runtime is recycled.

In [ ]:
!zip -qr results.zip results && ls -lh results.zip